In [1]:
# ============================================
# 1. Import Libraries
# ============================================
import pandas as pd

# ============================================
# 2. Load Dataset
# ============================================
df = pd.read_csv(
    "../data/SEED-ML/infertility_man_data-v2.csv",
    sep=";"
)

# ============================================
# 3. Define Target Variable
# ============================================
target = "diagnostic"
y = df[target]

# Display class distribution
print(y.value_counts())

diagnostic
NO                               6346
OLIGOASTHENOTHERATOZOOSPERMIA    1440
ASTHENOZOOSPERMIA                1180
THERATOZOOSPERMIA                 679
OLIGOZOOSPERMIA                   192
ASTHENOTHERATOZOOSPERMIA          140
OLIGOASTHENOZOOSPERMIA             97
OLIGOTHERATOZOOSPERMIA             34
AZOOSPERMIA                        16
Name: count, dtype: int64


/var/folders/qy/pmwwp9z11l9gx48kp84g7q640000gn/T/ipykernel_72758/2118610431.py:9: DtypeWarning: Columns (0: sample_vol_initial, 1: sample_concentration_initial, 2: sample_production_total, 3: sample_num_prog_mob_total, 4: sample_num_spz_normal_pre, 5: sample_morphology_normal_pre, 6: sample_num_spz_normal_post, 7: sample_morphology_normal_post) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [2]:
# ============================================
# 4. Define Feature Groups
# ============================================

# Core semen analysis variables (WHO-based)
macroscopic_microscopic_features = [
    "sample_state",
    "sample_appearance",
    "sample_agglutination",
    "sample_viscosity",
    "sample_liquefaction",
    "sample_red_blood_cells",
    "sample_leukocytes",
    "sample_ph",
    "sample_cells_round",
    "sample_vitality",
    "sample_survival_test",
    "sample_vol_initial",
    "sample_concentration_initial",
    "sample_morpho_normal",
    "sample_morpho_kruger",
    "sample_production_total",
    "sample_num_prog_mob_total",
    "sample_bodies_gelatinous",
]

# Detailed morphology variables
morphology_pre = [
    col for col in df.columns
    if col.endswith("_pre")
]

morphology_pre

['sample_num_spz_counted_pre',
 'sample_num_spz_normal_pre',
 'sample_morphology_normal_pre',
 'sample_heads_total_pre',
 'sample_heads_elongated_pre',
 'sample_heads_piriform_pre',
 'sample_heads_round_pre',
 'sample_heads_amorphous_pre',
 'sample_heads_macrocephalus_pre',
 'sample_heads_microcephalus_pre',
 'sample_heads_vacuole_pre',
 'sample_heads_small_acrosome_pre',
 'sample_heads_double_pre',
 'sample_heads_combined_pre',
 'sample_necks_bent_pre',
 'sample_necks_tails_total_pre',
 'sample_necks_ins_asymmetric_pre',
 'sample_necks_thick_pre',
 'sample_necks_thin_pre',
 'sample_necks_combined_pre',
 'sample_tails_short_pre',
 'sample_tails_broken_pre',
 'sample_tails_rolled_pre',
 'sample_tails_multiple_pre',
 'sample_tails_combined_pre',
 'sample_anormal_total_pre']

In [3]:
# ============================================
# 5. Define Additional Feature Groups
# ============================================

# Optional laboratory biomarkers
biomarkers = [
    "sample_scd",
    "sample_citric",
    "sample_fructose",
    "sample_spz_swollen",
]

# Post-treatment variables (excluded from baseline models)
post_treatment = [
    col for col in df.columns
    if col.endswith("_post") or col.endswith("_final")
]

In [6]:
# ============================================
# 6. Finalize Feature Groups
# ============================================

# Add treatment-related variables not captured by suffix matching
post_treatment += [
    "sample_recovery_technique",
    "sample_type_treat_recovery",
]

# Remove duplicate feature names while preserving order
post_treatment = list(dict.fromkeys(post_treatment))

# ============================================
# 7. Define Benchmark Feature Sets
# ============================================

tier1_features = macroscopic_microscopic_features

tier2_features = (
    macroscopic_microscopic_features
    + morphology_pre
)

tier3_features = (
    macroscopic_microscopic_features
    + morphology_pre
    + biomarkers
)

# Verify feature-set sizes and column availability
feature_sets = {
    "Tier 1": tier1_features,
    "Tier 2": tier2_features,
    "Tier 3": tier3_features,
}

for name, features in feature_sets.items():
    missing_columns = [col for col in features if col not in df.columns]

    print(f"{name}: {len(features)} features")
    print(f"Missing columns: {missing_columns}")
    print()

Tier 1: 18 features
Missing columns: []

Tier 2: 44 features
Missing columns: []

Tier 3: 48 features
Missing columns: []



In [7]:
# ============================================
# 8. Create Feature Inventory
# ============================================

feature_inventory = pd.DataFrame({
    "Variable": df.columns
})

feature_inventory["Role"] = "Other"

feature_inventory.loc[
    feature_inventory["Variable"] == target,
    "Role"
] = "Target"

feature_inventory.loc[
    feature_inventory["Variable"].isin(macroscopic_microscopic_features),
    "Role"
] = "Tier 1"

feature_inventory.loc[
    feature_inventory["Variable"].isin(morphology_pre),
    "Role"
] = "Morphology"

feature_inventory.loc[
    feature_inventory["Variable"].isin(biomarkers),
    "Role"
] = "Biomarker"

feature_inventory.loc[
    feature_inventory["Variable"].isin(post_treatment),
    "Role"
] = "Post-treatment / Exclude"

# Export feature inventory
feature_inventory.to_csv(
    "../results/feature_inventory.csv",
    index=False,
)

feature_inventory

,Variable,Role
0,diagnostic,Target
1,sample_state,Tier 1
2,sample_appearance,Tier 1
3,sample_agglutination,Tier 1
4,sample_viscosity,Tier 1
...,...,...
80,sample_anormal_total_post,Post-treatment / Exclude
81,sample_production_total_final,Post-treatment / Exclude
82,sample_citric,Biomarker
83,sample_fructose,Biomarker


In [8]:
# ============================================
# 9. Feature-Set Summary
# ============================================

print(f"Target: {target}")
print(f"Tier 1 features: {len(tier1_features)}")
print(f"Tier 2 features: {len(tier2_features)}")
print(f"Tier 3 features: {len(tier3_features)}")
print(f"Excluded post-treatment features: {len(post_treatment)}")

Target: diagnostic
Tier 1 features: 18
Tier 2 features: 44
Tier 3 features: 48
Excluded post-treatment features: 29
